In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
import xgboost as xgb
import joblib
import json
import os

print("All imports successful")

All imports successful


In [12]:
df = pd.read_csv('../data/processed/features_dataset_expanded.csv')

print(f"Expanded dataset loaded. Shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())

Expanded dataset loaded. Shape: (347296, 20)

Label distribution:
label
0    184844
1    162452
Name: count, dtype: int64


In [13]:
X = df.drop('label', axis=1)
y = df['label']
feature_names = X.columns.tolist()

print(f"Number of features: {len(feature_names)}")
print(f"Features: {feature_names}")

Number of features: 19
Features: ['url_length', 'domain_length', 'path_length', 'num_dots', 'num_hyphens', 'num_underscores', 'num_slashes', 'num_at_signs', 'num_question_marks', 'num_equals_signs', 'num_digits', 'uses_https', 'uses_ip_address', 'num_subdomains', 'has_www', 'has_suspicious_keyword', 'num_suspicious_keywords', 'domain_entropy', 'is_in_top_1million']


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} URLs")
print(f"Test set: {len(X_test)} URLs")

Training set: 277836 URLs
Test set: 69460 URLs


In [15]:
print("Training Random Forest on expanded dataset...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("Random Forest training complete!")

Training Random Forest on expanded dataset...
Random Forest training complete!


In [16]:
rf_predictions = rf_model.predict(X_test)

print("=== Random Forest Results (Expanded Dataset) ===")
acc  = accuracy_score(y_test, rf_predictions)
prec = precision_score(y_test, rf_predictions)
rec  = recall_score(y_test, rf_predictions)
f1   = f1_score(y_test, rf_predictions)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(
    y_test, rf_predictions,
    target_names=['Legitimate', 'Phishing']
))

=== Random Forest Results (Expanded Dataset) ===
Accuracy:  0.9985
Precision: 0.9998
Recall:    0.9970
F1 Score:  0.9984

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     36969
    Phishing       1.00      1.00      1.00     32491

    accuracy                           1.00     69460
   macro avg       1.00      1.00      1.00     69460
weighted avg       1.00      1.00      1.00     69460



In [17]:
print("Training XGBoost on expanded dataset...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, verbose=50)
print("XGBoost training complete!")

Training XGBoost on expanded dataset...
XGBoost training complete!


In [18]:
xgb_predictions = xgb_model.predict(X_test)

print("=== XGBoost Results (Expanded Dataset) ===")
acc  = accuracy_score(y_test, xgb_predictions)
prec = precision_score(y_test, xgb_predictions)
rec  = recall_score(y_test, xgb_predictions)
f1   = f1_score(y_test, xgb_predictions)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print()
print(classification_report(
    y_test, xgb_predictions,
    target_names=['Legitimate', 'Phishing']
))

=== XGBoost Results (Expanded Dataset) ===
Accuracy:  0.9985
Precision: 0.9997
Recall:    0.9970
F1 Score:  0.9984

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     36969
    Phishing       1.00      1.00      1.00     32491

    accuracy                           1.00     69460
   macro avg       1.00      1.00      1.00     69460
weighted avg       1.00      1.00      1.00     69460



In [19]:
# Compare old vs new results
results = {
    'Random Forest (Expanded)': {
        'Accuracy':  accuracy_score(y_test, rf_predictions),
        'Precision': precision_score(y_test, rf_predictions),
        'Recall':    recall_score(y_test, rf_predictions),
        'F1 Score':  f1_score(y_test, rf_predictions)
    },
    'XGBoost (Expanded)': {
        'Accuracy':  accuracy_score(y_test, xgb_predictions),
        'Precision': precision_score(y_test, xgb_predictions),
        'Recall':    recall_score(y_test, xgb_predictions),
        'F1 Score':  f1_score(y_test, xgb_predictions)
    }
}

comparison_df = pd.DataFrame(results).T
print("=== Expanded Model Comparison ===")
print(comparison_df.round(4).to_string())

best = comparison_df['F1 Score'].idxmax()
print(f"\nBest model: {best}")

=== Expanded Model Comparison ===
                          Accuracy  Precision  Recall  F1 Score
Random Forest (Expanded)    0.9985     0.9998   0.997    0.9984
XGBoost (Expanded)          0.9985     0.9997   0.997    0.9984

Best model: Random Forest (Expanded)


In [20]:
# Save the new expanded models
os.makedirs('../models/saved_models', exist_ok=True)

joblib.dump(rf_model, '../models/saved_models/random_forest_model.pkl')
joblib.dump(xgb_model, '../models/saved_models/xgboost_model.pkl')

with open('../models/saved_models/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

print("Expanded models saved to models/saved_models/")
print("The app and API will now use these improved models automatically")

Expanded models saved to models/saved_models/
The app and API will now use these improved models automatically
